In [ ]:
# ======================================================
# Notebook: Bayesian-style optimisation + Hyperparameter tuning
# Select next (10,2) inputs for noisy black-box maximisation
# ======================================================

import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor

# Load data
X = np.load("/mnt/data/initial_inputs.npy")     # (N,2)
y = np.load("/mnt/data/initial_outputs.npy")    # (N,)

# Hyperparameter tuning (surrogate model)
param_dist = {
    "n_estimators": [50,100,200,300],
    "max_depth": [None,3,5,8,12],
    "min_samples_split": [2,5,10],
    "min_samples_leaf": [1,2,4]
}

search = RandomizedSearchCV(
    RandomForestRegressor(),
    param_distributions=param_dist,
    n_iter=20,
    cv=3,
    random_state=42
)

search.fit(X, y)
model = search.best_estimator_

print("Best params:", search.best_params_)

# Candidate grid
x_min, x_max = X[:,0].min(), X[:,0].max()
y_min, y_max = X[:,1].min(), X[:,1].max()

gx = np.linspace(x_min, x_max, 100)
gy = np.linspace(y_min, y_max, 100)
XX, YY = np.meshgrid(gx, gy)
X_grid = np.vstack([XX.ravel(), YY.ravel()]).T

# Predict mean from ensemble
preds = np.array([tree.predict(X_grid) for tree in model.estimators_])
mean_pred = preds.mean(axis=0)
uncertainty = preds.std(axis=0)

# Acquisition = exploitation + exploration
acquisition = mean_pred + 0.5 * uncertainty

# Select next (10,2)
top_idx = np.argsort(acquisition)[-10:]
next_points = X_grid[top_idx]

print("Next (10,2) inputs:")
print(next_points)

# Plot
plt.figure(figsize=(6,5))
plt.contourf(XX, YY, mean_pred.reshape(XX.shape), levels=30)
plt.scatter(X[:,0], X[:,1])
plt.scatter(next_points[:,0], next_points[:,1], marker='x', s=100)
plt.show()